# An Image-Based Pipeline for BIPV Facade Area Estimation

An image-based pipeline for BIPV facade area estimation, converting street-level facade images into PVsyst-ready data.

This work develops an image-based pipeline for estimating usable BIPV facade area from street-level building images. The pipeline integrates object detection, segmentation, obstacle masking and removal, facade rectification, facade element segmentation, and real-world area estimation to generate PVsyst-ready structured outputs in JSON and Excel format.

This notebook runs the BIPV facade analysis project from GitHub:

https://github.com/DT-GAMER/BIPV_Project

It supports:

- single building image analysis
- multiple image batch analysis
- workflow figure output like the methodology diagram
- focused 8-stage image-based facade area workflow
- JSON and Excel export for downstream reporting

Use a GPU runtime before running:

`Runtime -> Change runtime type -> GPU`

## 1. Mount Google Drive

Your input images and exported results will be read/written from Google Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

## 1B. Google Drive Folder Setup

Each person running this notebook needs their own input/output folder in Google Drive.
They do not need to create the project code files in Drive; Colab clones the GitHub repo into `/content/BIPV_Project` automatically.

Recommended Drive structure:

```text
MyDrive/
  BIPV_Project/
    IMG_5976.jpeg
    IMG_6152.jpeg
    IMG_6133.jpeg
    IMG_5978.jpeg
    outputs/
```

Steps:

1. Open Google Drive.
2. Create a folder named `BIPV_Project` inside `MyDrive`.
3. Upload building facade images into that folder.
4. This notebook will create/use `outputs/` for JSON files and saved result images.

If your folder or filenames are different, update `IMAGE_PATH`, `IMAGE_PATHS`, `OUTPUT_DIR`, and `OUTPUT_PATH` in Step 6.


In [ ]:
import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/BIPV_Project'
OUTPUT_DIR = f'{DRIVE_PROJECT_DIR}/outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Drive project folder:', DRIVE_PROJECT_DIR)
print('Output folder:', OUTPUT_DIR)
print('Upload your input facade images into:', DRIVE_PROJECT_DIR)


## 2. Clone or Update the GitHub Project

Run this cell every time you start a fresh Colab runtime. If the project already exists, it pulls the latest version.

In [ ]:
import os

REPO_URL = 'https://github.com/DT-GAMER/BIPV_Project.git'
PROJECT_DIR = '/content/BIPV_Project'

if not os.path.exists(PROJECT_DIR):
    %cd /content
    !git clone {REPO_URL}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!ls

## 3. Install Dependencies

This project uses SAM, Grounding DINO, LaMa, OpenCV, scikit-image, PyTorch, and related packages.

After installing, Colab may require a runtime restart. If so, restart and continue from Step 4.

In [ ]:
%cd /content/BIPV_Project
!pip install -q -r requirements-colab.txt
print('Dependencies installed. If Colab asks for a restart, restart runtime now.')

## 4. Fix NumPy Binary Compatibility

Run this section to fix:

`ValueError: numpy.dtype size changed, may indicate binary incompatibility`

After this cell, restart the runtime.

In [ ]:
# FIX NumPy/scikit-image binary compatibility breaks.
%cd /content/BIPV_Project
!pip uninstall -y numpy scipy scikit-image opencv-python opencv-python-headless pandas
!pip install --no-cache-dir --force-reinstall numpy==1.26.4 scipy==1.13.1 scikit-image==0.24.0 opencv-python-headless==4.10.0.84 pandas==2.2.2 matplotlib==3.9.2
print('Restart runtime after this cell.')

## 5. Import the Project

Run this after installing dependencies or after every runtime restart.

In [ ]:
%cd /content/BIPV_Project

import sys, importlib
sys.path = [p for p in sys.path if p != '/content/BIPV_Project/src']
sys.path.insert(0, '/content/BIPV_Project')
importlib.invalidate_caches()

from src.config import automatic_config, AnalysisConfig
from src.pipeline import run_bipv_analysis
from src.batch import run_batch_analysis
from src.visualization import (
    show_workflow_grid,
    workflow_images_from_result,
)

print('Imports working')

## 5B. Check GPU / CUDA

This project needs a GPU-enabled PyTorch runtime. If this prints `CUDA available: False`, switch Colab to GPU runtime and avoid reinstalling CPU-only torch.

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA is not available. Go to Runtime -> Change runtime type -> GPU, then restart runtime.")

## 6. Configure Input Images

Put your building images in Google Drive. Update the paths below to match your files.

For one image, use `IMAGE_PATH`.

For multiple images, use `IMAGE_PATHS`.

In [ ]:
# Update these filenames to match the images uploaded to your Google Drive folder.
IMAGE_PATH = f'{DRIVE_PROJECT_DIR}/IMG_5976.jpeg'

IMAGE_PATHS = [
    f'{DRIVE_PROJECT_DIR}/IMG_5976.jpeg',
    f'{DRIVE_PROJECT_DIR}/IMG_6152.jpeg',
    f'{DRIVE_PROJECT_DIR}/IMG_6133.jpeg',
    f'{DRIVE_PROJECT_DIR}/IMG_5978.jpeg',
]

OUTPUT_PATH = f'{DRIVE_PROJECT_DIR}/pvsyst_export.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Single image:', IMAGE_PATH)
print('Batch images:', len(IMAGE_PATHS))
print('Output directory:', OUTPUT_DIR)

## 7A. Run One Image - Automatic Mode

This mode requires no manual dimensions. The model estimates facade dimensions automatically from image/floor/window evidence.

The output area is therefore an **estimated usable BIPV area**, not a measured ground-truth value.

In [ ]:
config = automatic_config(
    image_path=IMAGE_PATH,
    output_path=OUTPUT_PATH,
)

result = run_bipv_analysis(config)

print('Done')
print('JSON export saved to:', result['output_path'])
print('Excel export saved to:', result['excel_output_path'])

## 7B. Optional Calibrated Mode

Use this only when you have measured facade width and height from Google Earth, GIS, architectural drawings, or another reliable source.

Skip this section for fully automatic image-only mode.

In [ ]:
# Optional example. Uncomment and edit values if measured dimensions are available.
# config = AnalysisConfig(
#     image_path=IMAGE_PATH,
#     output_path=OUTPUT_PATH,
#     ge_width_m=42.5,
#     ge_height_m=18.0,
#     require_google_earth_dimensions=True,
# )
# result = run_bipv_analysis(config)

## 8. View One-Image Workflow Result

This shows the five rows used in the methodology figure:

1. Facade Image
2. Obstacle Detection
3. Obstacle Removal
4. Facade Alignment
5. Segmentation Result

In [ ]:
show_workflow_grid(result, column_titles=['Single Building'])

## 9. Print One-Image Engineering Summary

In [ ]:
print('Dimensions:', result['dimensions'])
print('Rectification:', result['rectification'])
print('Scaling:', result['stages']['scaling'])
print('Usable area m²:', result['usable_results']['usable_area_m2'])
print('Panel capacity:', result['panel_capacity'])
print('Energy yield:', result['energy_yield'])

## 10. Current Methodology Focus

Shadow and illumination analysis is currently disabled.

The active workflow focuses on image acquisition, object detection, SAM segmentation, obstacle masking/removal, facade rectification, facade element segmentation, and usable BIPV area estimation.

In [ ]:
print('Active stage count: 8')
print('Focus: facade area estimation without shadow/illumination analysis')

## 11. Run Multiple Images at Once

Batch mode reuses the loaded models, so it is better than running the notebook from scratch for each image.

In [ ]:
MAX_BATCH_IMAGES = 10

if len(IMAGE_PATHS) > MAX_BATCH_IMAGES:
    raise ValueError(f'You provided {len(IMAGE_PATHS)} images. Maximum allowed per batch is {MAX_BATCH_IMAGES}. Split into smaller batches.')

results = run_batch_analysis(
    IMAGE_PATHS,
    output_dir=OUTPUT_DIR,
    max_images=MAX_BATCH_IMAGES,
)

print('Batch complete:', len(results), 'images')

## 12. Display Batch Workflow Grid

In [ ]:
column_titles = ['IMG_5976', 'IMG_6152', 'IMG_6133', 'IMG_5978']
show_workflow_grid(results, column_titles=column_titles)

## 13. Save Workflow Grid as JPG

Use this to create a single image you can send to someone.

In [ ]:
from src.visualization import save_workflow_grid_image, show_workflow_grid

SAVE_PATH = f'{OUTPUT_DIR}/workflow_results.jpg'
save_workflow_grid_image(results, SAVE_PATH, column_titles=column_titles, dpi=300)

# Display it in the notebook too.
show_workflow_grid(results, column_titles=column_titles)

print('Saved to:', SAVE_PATH)

## 14. Print Batch Summary

In [ ]:
for image_path, r in zip(IMAGE_PATHS, results):
    print('\n---', image_path, '---')
    print('Dimensions:', r['dimensions'])
    print('Usable area m²:', r['usable_results']['usable_area_m2'])
    print('Panel capacity:', r['panel_capacity'])
    print('Annual kWh:', r['energy_yield']['annual_kwh'])
    print('JSON export:', r['output_path'])
    print('Excel export:', r['excel_output_path'])

## 15. Train Facade Parser Model (Optional Research Upgrade)

Use this section only when you want to train the facade-specific segmentation model from your Roboflow dataset. This improves the red window/opening masks over time.

Run this in a **GPU runtime**:

`Runtime -> Change runtime type -> GPU`

The current Roboflow dataset is small, so the first model is a prototype. More labelled images will be needed for calculation-grade accuracy.


## 15A. Install Training Dependencies

This installs Ultralytics YOLO and the light training utilities.


In [ ]:
%cd /content/BIPV_Project
!pip install -q -r requirements-training.txt


## 15B. Confirm GPU Is Available

If this says CUDA is not available, change the Colab runtime to GPU before training.


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU is not available. Go to Runtime -> Change runtime type -> GPU.')


## 15C. Copy Roboflow Dataset Into The Project

Put your downloaded Roboflow dataset in Google Drive first. You can use either:

1. A ZIP file, for example `/content/drive/MyDrive/BIPV_Project/bipv-facade-parsing.v1i.yolov8.zip`
2. An extracted folder, for example `/content/drive/MyDrive/BIPV_Project/bipv-facade-parsing.v1i.yolov8`

This cell copies it into:

`/content/BIPV_Project/training/datasets/facade_parser/`


In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT_ROOT = Path('/content/BIPV_Project')
DATASET_TARGET = PROJECT_ROOT / 'training/datasets/facade_parser'

# Update one of these paths if your Drive location is different.
DRIVE_DATASET_ZIP = Path('/content/drive/MyDrive/BIPV_Project/bipv-facade-parsing.v1i.yolov8.zip')
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/BIPV_Project/bipv-facade-parsing.v1i.yolov8')

TEMP_EXTRACT_DIR = Path('/content/roboflow_facade_dataset')

if DATASET_TARGET.exists():
    shutil.rmtree(DATASET_TARGET)
DATASET_TARGET.mkdir(parents=True, exist_ok=True)

if DRIVE_DATASET_ZIP.exists():
    if TEMP_EXTRACT_DIR.exists():
        shutil.rmtree(TEMP_EXTRACT_DIR)
    TEMP_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(TEMP_EXTRACT_DIR)
    candidates = [p for p in TEMP_EXTRACT_DIR.iterdir() if p.is_dir()]
    source_root = candidates[0] if len(candidates) == 1 else TEMP_EXTRACT_DIR
elif DRIVE_DATASET_DIR.exists():
    source_root = DRIVE_DATASET_DIR
else:
    raise FileNotFoundError(
        'Could not find the Roboflow dataset ZIP or folder in Drive. '
        'Update DRIVE_DATASET_ZIP or DRIVE_DATASET_DIR.'
    )

split_map = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}
for source_split, target_split in split_map.items():
    image_src = source_root / source_split / 'images'
    label_src = source_root / source_split / 'labels'
    if not image_src.exists() or not label_src.exists():
        continue

    image_dst = DATASET_TARGET / 'images' / target_split
    label_dst = DATASET_TARGET / 'labels' / target_split
    image_dst.mkdir(parents=True, exist_ok=True)
    label_dst.mkdir(parents=True, exist_ok=True)

    for item in image_src.iterdir():
        if item.is_file():
            shutil.copy2(item, image_dst / item.name)
    for item in label_src.iterdir():
        if item.is_file():
            shutil.copy2(item, label_dst / item.name)

for extra_name in ['data.yaml', 'README.roboflow.txt', 'README.dataset.txt']:
    extra = source_root / extra_name
    if extra.exists():
        shutil.copy2(extra, DATASET_TARGET / extra.name)

for split in ['train', 'val', 'test']:
    images = list((DATASET_TARGET / 'images' / split).glob('*'))
    labels = list((DATASET_TARGET / 'labels' / split).glob('*.txt'))
    print(f'{split}: {len(images)} images, {len(labels)} labels')


## 15D. Verify Dataset Labels

This checks that every image has a matching YOLO segmentation label file.


In [1]:
from pathlib import Path

root = Path('/content/BIPV_Project/training/datasets/facade_parser')
for split in ['train', 'val', 'test']:
    image_dir = root / 'images' / split
    label_dir = root / 'labels' / split
    images = {p.stem for p in image_dir.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    print(split)
    print('  images:', len(images))
    print('  labels:', len(labels))
    print('  missing labels:', sorted(images - labels))
    print('  orphan labels:', sorted(labels - images))


train
  images: 0
  labels: 0
  missing labels: []
  orphan labels: []
val
  images: 0
  labels: 0
  missing labels: []
  orphan labels: []
test
  images: 0
  labels: 0
  missing labels: []
  orphan labels: []


## 15E. Train YOLO Segmentation Facade Parser

The best weights will be saved under:

`training/runs/facade_parser_yolo/weights/best.pt`


In [ ]:
%cd /content/BIPV_Project

!python scripts/train_facade_parser.py   --data training/facade_parser_dataset.yaml   --model yolo11s-seg.pt   --epochs 100   --imgsz 1024   --batch 4


## 15F. Evaluate The Trained Model

Use the validation split first. With only a small dataset, metrics may be unstable, but this confirms the model and labels are working.


In [ ]:
%cd /content/BIPV_Project

from pathlib import Path

def find_facade_parser_weights():
    candidates = [
        Path('/content/BIPV_Project/training/runs/facade_parser_yolo/weights/best.pt'),
        Path('/content/BIPV_Project/runs/segment/training/runs/facade_parser_yolo/weights/best.pt'),
    ]
    candidates.extend(Path('/content/BIPV_Project').glob('**/facade_parser_yolo/weights/best.pt'))
    existing = [path for path in candidates if path.exists()]
    if not existing:
        raise FileNotFoundError('Could not find best.pt. Run 15E first and check the training output path.')
    return max(existing, key=lambda path: path.stat().st_mtime)

WEIGHTS_PATH = find_facade_parser_weights()
print('Using weights:', WEIGHTS_PATH)

!python scripts/evaluate_facade_parser.py \
  --weights {WEIGHTS_PATH} \
  --data training/facade_parser_dataset.yaml \
  --split val


## 15G. Save The Best Weights To Google Drive

This copies the trained model to Drive so it is not lost when the Colab runtime resets.

The trained parser is currently disabled by default. The main pipeline uses Grounding DINO + SAM unless you explicitly set `use_trained_facade_parser=True`.


In [ ]:
from pathlib import Path
import shutil

def find_facade_parser_weights():
    candidates = [
        Path('/content/BIPV_Project/training/runs/facade_parser_yolo/weights/best.pt'),
        Path('/content/BIPV_Project/runs/segment/training/runs/facade_parser_yolo/weights/best.pt'),
    ]
    candidates.extend(Path('/content/BIPV_Project').glob('**/facade_parser_yolo/weights/best.pt'))
    existing = [path for path in candidates if path.exists()]
    if not existing:
        raise FileNotFoundError('Could not find best.pt. Run 15E first and check the training output path.')
    return max(existing, key=lambda path: path.stat().st_mtime)

WEIGHTS_PATH = find_facade_parser_weights()
DRIVE_MODEL_PATH = Path('/content/drive/MyDrive/BIPV_Project/models/facade_parser.pt')
DRIVE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(WEIGHTS_PATH, DRIVE_MODEL_PATH)
print('Copied from:', WEIGHTS_PATH)
print('Saved trained model to:', DRIVE_MODEL_PATH)


### After Saving: Normal Pipeline Uses It Automatically

The saved model remains available at `/content/drive/MyDrive/BIPV_Project/models/facade_parser.pt`, but the normal analysis flow does not use it by default. To experiment with it later, create an `AnalysisConfig` with `use_trained_facade_parser=True`. If the trained model is missing or unusable, the code falls back to Grounding DINO + SAM.


## 15H. Test Predictions Visually

This creates prediction images so you can inspect whether windows/openings are being detected correctly.


In [ ]:
%cd /content/BIPV_Project

!python scripts/predict_facade_parser.py   --weights training/runs/facade_parser_yolo/weights/best.pt   --source training/datasets/facade_parser/images/val   --project outputs/facade_parser_predictions   --name val_predictions

print('Prediction outputs saved under /content/BIPV_Project/outputs/facade_parser_predictions')


## 99. End Training Section
